# AccentShift — Seed-VC Fine-Tuning on Kaggle

**Before running:**
1. Settings (right panel) → **Accelerator: GPU T4 x2** or P100
2. Settings → **Internet: On**
3. Add-ons → **Secrets** → add `TELEGRAM_TOKEN` and `TELEGRAM_CHAT_ID` (optional but recommended)
4. Run All

Expected total time: ~2–3 hours on T4, ~3–5 hours on P100.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# Cell 2: Telegram setup (reads from Kaggle Secrets)
# Add-ons → Secrets → add TELEGRAM_TOKEN and TELEGRAM_CHAT_ID before running
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['TELEGRAM_TOKEN'] = secrets.get_secret('TELEGRAM_TOKEN')
    os.environ['TELEGRAM_CHAT_ID'] = secrets.get_secret('TELEGRAM_CHAT_ID')
    print('Telegram secrets loaded OK')
except Exception as e:
    print(f'Telegram secrets not found ({e}) — training will run without notifications')

In [ ]:
%%bash
# Cell 3: Clone repo
echo '=== Cloning AccentShift repo ==='
git clone --depth 1 https://github.com/nischal2805/AccentShift.git /kaggle/working/AccentShift
cd /kaggle/working/AccentShift
git checkout div
echo "Branch: $(git branch --show-current)"
echo "Latest commit: $(git log --oneline -1)"
echo '=== Repo ready ==='

In [ ]:
%%bash
# Cell 4: Install pipeline deps (torch already on Kaggle)
echo '=== Installing deps ==='
pip install -q \
    "librosa>=0.10.2" soundfile pyloudnorm pyworld silero-vad \
    transformers jiwer speechbrain scikit-learn joblib pyyaml click \
    huggingface-hub tqdm hydra-core omegaconf einops munch \
    accelerate pydub tensorboard gdown 2>&1 | tail -5
echo '=== Deps installed ==='

In [ ]:
%%bash
# Cell 5: Clone Seed-VC + Amphion + install seed-vc requirements
cd /kaggle/working/AccentShift/backend
mkdir -p third_party

echo '=== Cloning Seed-VC ==='
if [ ! -d third_party/seed-vc ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/Plachtaa/seed-vc.git third_party/seed-vc
    echo 'Seed-VC cloned'
else
    echo 'Seed-VC already present'
fi

echo '=== Installing Seed-VC deps ==='
pip install -q -r third_party/seed-vc/requirements.txt --no-deps 2>/dev/null || true
echo 'Seed-VC deps done'

echo '=== Cloning Amphion ==='
if [ ! -d third_party/Amphion ]; then
    git clone --depth 1 --filter=blob:none --single-branch \
        https://github.com/open-mmlab/Amphion.git third_party/Amphion
    echo 'Amphion cloned'
else
    echo 'Amphion already present'
fi

echo '=== third_party ready ==='

In [ ]:
%%bash
# Cell 6: Download L2-Arctic v5.0 from Google Drive (~7GB)
echo '=== Downloading L2-Arctic dataset ==='
mkdir -p /kaggle/working/AccentShift/backend/data/l2arctic_raw
ZIP=/kaggle/working/AccentShift/backend/data/l2arctic_raw/l2arctic_v5.zip

if [ ! -f "$ZIP" ]; then
    gdown 'https://drive.google.com/uc?id=1ciCw_ttbw7a9r7d5DZzTJwoZq5rQB3TA' -O "$ZIP"
else
    echo 'ZIP already downloaded, skipping'
fi
echo "ZIP size: $(du -sh $ZIP)"

In [ ]:
%%bash
# Cell 7: Unzip + organize WAVs by accent
# L2-Arctic v5 contains per-speaker zips (ABA.zip, ASI.zip...) inside the main zip
cd /kaggle/working/AccentShift/backend
RAW=data/l2arctic_raw

echo '=== Unzipping main archive ==='
unzip -o -q $RAW/l2arctic_v5.zip -d $RAW
echo 'Main zip extracted'

echo '=== Unzipping per-speaker archives ==='
for spk_zip in $RAW/*.zip; do
    [ "$spk_zip" = "$RAW/l2arctic_v5.zip" ] && continue
    [ -f "$spk_zip" ] || continue
    spk=$(basename "$spk_zip" .zip)
    echo "  $spk..."
    unzip -o -q "$spk_zip" -d $RAW
done
echo 'All speakers extracted'

echo '=== Organizing WAVs by accent ==='
bash scripts/download_l2arctic.sh

echo ''
echo '=== WAV counts per accent ==='
for d in data/finetune/*/; do
    [ -d "$d" ] || continue
    echo "  $(basename $d): $(find $d -name '*.wav' | wc -l) WAVs"
done

In [ ]:
%%bash
# Cell 8: TRAIN — ~2-3 hours on T4
# fp16 = T4/P100 compatible (bf16 requires A100/L40S)
# Telegram notifications sent if secrets are loaded
# First step downloads HuBERT + CAMPPlus from HF (~2GB) — normal, not a hang
cd /kaggle/working/AccentShift/backend
export HF_HOME=/kaggle/working/AccentShift/backend/.hf_cache
export TELEGRAM_TOKEN=$TELEGRAM_TOKEN
export TELEGRAM_CHAT_ID=$TELEGRAM_CHAT_ID

echo '=== Starting training ==='
echo "Time: $(date)"
echo ''

python scripts/finetune_style.py \
    --accent all \
    --steps 15000 \
    --batch-size 8 \
    --save-every 1000 \
    --num-workers 2 \
    --mixed-precision fp16

echo ''
echo "=== Training finished at $(date) ==="

In [ ]:
%%bash
# Cell 9: Verify checkpoints
echo '=== Checkpoints ==='
find /kaggle/working/AccentShift/backend/runs/ -name '*.pth' -exec du -sh {} \;
echo ''
echo '=== Run directory listing ==='
ls -lh /kaggle/working/AccentShift/backend/runs/*/ 2>/dev/null || echo 'No runs/ directory found'

In [ ]:
import shutil, os
# Cell 10: Zip checkpoint for download
runs_dir = '/kaggle/working/AccentShift/backend/runs/'
out_zip = '/kaggle/working/seedvc_finetuned'
shutil.make_archive(out_zip, 'zip', runs_dir)
size = os.path.getsize(out_zip + '.zip') / (1024**2)
print(f'Checkpoint zipped: {out_zip}.zip ({size:.0f} MB)')
print('Download via Files panel on the right ->')